<a href="https://colab.research.google.com/github/CemKeremSahin/NIR-to-RGB-Colorization/blob/main/Notebooks/S_NET_3Channels_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
print(f"Sahip olduğun CPU çekirdek sayısı: {os.cpu_count()}")

In [ ]:
# --- 1. COLAB KURULUMLARI VE DRIVE BAĞLANTISI ---
import os

# Gerekli kütüphaneyi kuruyoruz
try:
    import segmentation_models_pytorch as smp
    print("Kütüphane zaten kurulu.")
except ImportError:
    print("Kütüphane kuruluyor...")
    os.system('pip install segmentation_models_pytorch')
    import segmentation_models_pytorch as smp

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Drive dan colab e aktarma
!cp -r /content/drive/MyDrive/4sensor_dataset/low_light_20230117/train /content/

In [ ]:
import os

# Görselinize göre güncellenmiş klasör yolları
DIR_785 = "/content/train/785nm"
DIR_850 = "/content/train/850nm"
DIR_940 = "/content/train/940nm"
RGB_FOLDER = "/content/train/gt_rgb"

print("--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---\n")

def veri_kontrol_3ch():
    klasorler = {
        "785nm": DIR_785,
        "850nm": DIR_850,
        "940nm": DIR_940,
        "GT_RGB": RGB_FOLDER
    }

    dosya_listeleri = {}

    # Tüm klasörlerin varlığını ve dosya sayılarını kontrol et
    for isim, yol in klasorler.items():
        if not os.path.exists(yol):
            print(f"HATA: {yol} klasörü bulunamadı! Lütfen yolu kontrol edin.")
            return

        # .bmp, .png veya .jpg uzantılı dosyaları okuyacak şekilde genişlettik
        dosyalar = sorted([f for f in os.listdir(yol) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        dosya_listeleri[isim] = dosyalar
        print(f"{isim} klasöründe {len(dosyalar)} adet görüntü dosyası bulundu.")

    # Tüm klasörlerdeki dosya sayılarının eşit olup olmadığını kontrol et
    uzunluklar = [len(liste) for liste in dosya_listeleri.values()]
    if len(set(uzunluklar)) == 1:
        print("\n>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.")
    else:
        print("\n>>> UYARI: Dosya sayıları UYUŞMUYOR!")
        for isim, liste in dosya_listeleri.items():
            print(f"    {isim}: {len(liste)}")

    # Önceki kodunuzdaki 538 sayısı Real-World veri setinin toplamıdır[cite: 387].
    # Klasörünüzdeki sayı 458 çıkarsa bu normaldir çünkü 538 görüntünün 458'i eğitim (train), 80'i test içindir[cite: 388].

    # İlk 5 eşleşmeyi yan yana yazdırarak kontrol et
    print("\nÖrnek Eşleşmeler (İlk 5):")
    min_len = min(uzunluklar) if uzunluklar else 0

    if min_len > 0:
        for i in range(min(5, min_len)):
            f_785 = dosya_listeleri["785nm"][i]
            f_850 = dosya_listeleri["850nm"][i]
            f_940 = dosya_listeleri["940nm"][i]
            f_rgb = dosya_listeleri["GT_RGB"][i]
            print(f"  {f_785} | {f_850} | {f_940}  <--*-->  {f_rgb}")
    else:
        print("Klasörler boş olduğu için örnek gösterilemiyor.")

veri_kontrol_3ch()

--- 3 KANALLI (MULTI-BAND) VERİ SETİ KONTROLÜ BAŞLIYOR ---

785nm klasöründe 206 adet görüntü dosyası bulundu.
850nm klasöründe 206 adet görüntü dosyası bulundu.
940nm klasöründe 206 adet görüntü dosyası bulundu.
GT_RGB klasöründe 206 adet görüntü dosyası bulundu.

>>> BAŞARILI: Tüm NIR bantları (785, 850, 940) ve RGB dosya sayıları EŞİT.

Örnek Eşleşmeler (İlk 5):
  001.bmp | 001.bmp | 001.bmp  <--*-->  0001.bmp
  002.bmp | 002.bmp | 002.bmp  <--*-->  0002.bmp
  003.bmp | 003.bmp | 003.bmp  <--*-->  0003.bmp
  004.bmp | 004.bmp | 004.bmp  <--*-->  0004.bmp
  005.bmp | 005.bmp | 005.bmp  <--*-->  0005.bmp


In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models
from tqdm import tqdm
from google.colab import runtime

# --- 1. PERFORMANS VE CİHAZ AYARLARI ---
#torch.backends.cuda.matmul.fp32_precision = 'high'
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 2. AYARLAR VE YOLLAR ---
ROOT_PATH = "/content/train"
SAVE_PATH = "/content/drive/MyDrive/Computer_Vision/Model_Weights/S-NET_3Channels"
os.makedirs(SAVE_PATH, exist_ok=True)

PATCH_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 3500
SAVE_EVERY_N_EPOCHS = 500

# --- 3. S-NET MİMARİSİ (GENERATOR) ---
# S-Net, genellikle ardışıl evrişim blokları ve simetrik bir yapıdan oluşur.
class SNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super(SNetBlock, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class SNetGenerator(nn.Module):
    """
    Dong et al. tarafından önerilen S-Shape Network yaklaşımı.
    U-Net ile benzer parametre sayısına (56M) sahiptir.
    """
    def __init__(self, in_channels=3, out_channels=3): # in_channels 3 yapıldı (3 NIR bandı için)
        super(SNetGenerator, self).__init__()

        # Encoder (Daralan Yol)
        self.e1 = SNetBlock(in_channels, 64)
        self.e2 = SNetBlock(64, 128, stride=2)
        self.e3 = SNetBlock(128, 256, stride=2)
        self.e4 = SNetBlock(256, 512, stride=2)
        self.e5 = SNetBlock(512, 1024, stride=2)

        # Decoder (Genişleyen Yol - S Şekli)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        self.d4 = SNetBlock(1024 + 512, 512)
        self.d3 = SNetBlock(512 + 256, 256)
        self.d2 = SNetBlock(256 + 128, 128)
        self.d1 = SNetBlock(128 + 64, 64)

        self.final = nn.Sequential(
            nn.Conv2d(64, out_channels, kernel_size=1),
            nn.Tanh() # Görüntü aralığını [-1, 1] yapmak için
        )

    def forward(self, x):
        # Encoder aşaması
        s1 = self.e1(x)
        s2 = self.e2(s1)
        s3 = self.e3(s2)
        s4 = self.e4(s3)
        s5 = self.e5(s4)

        # Decoder ve Skip Connections (S-Shape akışı)
        x = self.d4(torch.cat([self.up(s5), s4], dim=1))
        x = self.d3(torch.cat([self.up(x), s3], dim=1))
        x = self.d2(torch.cat([self.up(x), s2], dim=1))
        x = self.d1(torch.cat([self.up(x), s1], dim=1))

        return self.final(x)

# --- 4. DISCRIMINATOR (PATCHGAN) ---
class PatchGAN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(in_f, out_f, stride=2, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride, 1)]
            if norm: layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.model = nn.Sequential(
            block(6, 64, norm=False), # in_f 4'ten 6'ya çıkarıldı (3 IR + 3 RGB)
            block(64, 128),
            block(128, 256),
            block(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, 1, 1)
        )
    def forward(self, ir, rgb):
        return self.model(torch.cat((ir, rgb), 1))

# --- 5. DATASET VE KAYIP FONKSİYONLARI ---
class MultiBandNIRDataset(Dataset):
    def __init__(self, root_path, patch_size=256):
        self.patch_size = patch_size
        self.data_cache = []

        dir_785 = os.path.join(root_path, "785nm")
        dir_850 = os.path.join(root_path, "850nm")
        dir_940 = os.path.join(root_path, "940nm")
        rgb_dir = os.path.join(root_path, "gt_rgb") # Senin attığın orijinal koddaki klasör adı

        files_785 = sorted([f for f in os.listdir(dir_785) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_850 = sorted([f for f in os.listdir(dir_850) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        files_940 = sorted([f for f in os.listdir(dir_940) if f.lower().endswith(('.bmp', '.png', '.jpg'))])
        rgb_files = sorted([f for f in os.listdir(rgb_dir) if f.lower().endswith(('.bmp', '.png', '.jpg'))])

        print(f"📥 Veriler yükleniyor...")
        for f7, f8, f9, fr in tqdm(zip(files_785, files_850, files_940, rgb_files), total=len(files_785)):

            img_785 = cv2.imread(os.path.join(dir_785, f7), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_850 = cv2.imread(os.path.join(dir_850, f8), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
            img_940 = cv2.imread(os.path.join(dir_940, f9), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

            # 3 bandı üst üste ekliyoruz. Şekil: (H, W, 3)
            ir_img = np.stack([img_785, img_850, img_940], axis=-1)

            rgb_img = cv2.imread(os.path.join(rgb_dir, fr), cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(rgb_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

            # np.expand_dims KULLANMIYORUZ, çünkü zaten ir_img (H, W, 3) boyutunda
            self.data_cache.append({'ir': ir_img, 'rgb': rgb_img})

    def __len__(self): return len(self.data_cache)

    def __getitem__(self, idx):
        d = self.data_cache[idx]
        ir = torch.from_numpy(d['ir']).permute(2, 0, 1) * 2.0 - 1.0
        rgb = torch.from_numpy(d['rgb']).permute(2, 0, 1) * 2.0 - 1.0
        i, j, h, w = transforms.RandomCrop.get_params(ir, (self.patch_size, self.patch_size))
        return transforms.functional.crop(ir, i, j, h, w), transforms.functional.crop(rgb, i, j, h, w)

class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Makalede de kullanılan VGG tabanlı özellik kaybı
        self.vgg = models.vgg16(weights='DEFAULT').features[:16].eval().to(DEVICE)
        for p in self.vgg.parameters(): p.requires_grad = False
        self.criterion = nn.L1Loss()
    def forward(self, fake, real):
        return self.criterion(self.vgg((fake+1)/2), self.vgg((real+1)/2))

# --- 6. EĞİTİM DÖNGÜSÜ ---
model_G = SNetGenerator(3, 3).to(DEVICE) # 1 yerine 3 yapıldı
model_D = PatchGAN().to(DEVICE)

optimizer_G = optim.Adam(model_G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optimizer_D = optim.Adam(model_D.parameters(), lr=2e-4, betas=(0.5, 0.999))

# --- AMP İÇİN SCALER EKLEME ---
scaler_G = torch.cuda.amp.GradScaler()
scaler_D = torch.cuda.amp.GradScaler()

criterion_GAN = nn.BCEWithLogitsLoss()
criterion_MAE = nn.L1Loss()
criterion_percep = PerceptualLoss()

loader = DataLoader(MultiBandNIRDataset(ROOT_PATH, PATCH_SIZE), BATCH_SIZE, shuffle=True, pin_memory=True)

print(f"\n🚀 3 Kanallı S-Net Eğitimi Başlıyor (AMP Aktif)...")

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g, epoch_d = 0, 0
    loop = tqdm(loader, leave=False)
    for ir, real in loop:
        ir, real = ir.to(DEVICE), real.to(DEVICE)

        # --- DISCRIMINATOR EĞİTİMİ (AMP) ---
        optimizer_D.zero_grad()
        with torch.cuda.amp.autocast(): # 16-bit hassasiyet başlangıcı
            fake = model_G(ir).detach()
            l_d_real = criterion_GAN(model_D(ir, real), torch.ones_like(model_D(ir, real)))
            l_d_fake = criterion_GAN(model_D(ir, fake), torch.zeros_like(model_D(ir, fake)))
            l_d = (l_d_real + l_d_fake) * 0.5

        # Ölçeklendirilmiş geri yayılım
        scaler_D.scale(l_d).backward()
        scaler_D.step(optimizer_D)
        scaler_D.update()

        # --- GENERATOR EĞİTİMİ (AMP) ---
        optimizer_G.zero_grad()
        with torch.cuda.amp.autocast(): # 16-bit hassasiyet başlangıcı
            fake = model_G(ir)
            l_g_mae = criterion_MAE(fake, real) * 100.0
            l_g_percep = criterion_percep(fake, real) * 10.0
            l_g_gan = criterion_GAN(model_D(ir, fake), torch.ones_like(model_D(ir, fake)))
            l_g = l_g_gan + l_g_mae + l_g_percep

        # Ölçeklendirilmiş geri yayılım
        scaler_G.scale(l_g).backward()
        scaler_G.step(optimizer_G)
        scaler_G.update()

        epoch_g += l_g.item(); epoch_d += l_d.item()
        loop.set_description(f"Epoch [{epoch}/{NUM_EPOCHS}]")
        loop.set_postfix(G=l_g.item(), D=l_d.item())

    # Kayıt ve Log kısımları aynı kalabilir...

    if epoch % 100 == 0: print(f"Epoch {epoch} | G_Loss: {epoch_g/len(loader):.4f}")

    if epoch % SAVE_EVERY_N_EPOCHS == 0:
      torch.save({
        'epoch': epoch,
        'model_G_state_dict': model_G.state_dict(),
        'model_D_state_dict': model_D.state_dict(),
        'optimizer_G_state_dict': optimizer_G.state_dict(),
        'optimizer_D_state_dict': optimizer_D.state_dict(),
        # --- AMP İÇİN KRİTİK ---
        'scaler_G_state_dict': scaler_G.state_dict(),
        'scaler_D_state_dict': scaler_D.state_dict(),
      }, os.path.join(SAVE_PATH, f"S-Net_3Channels_epoch_{epoch}.pth"))

print("✅ Eğitim tamamlandı.")

print("✅ Eğitim bitti. Oturum otomatik olarak kapatılıyor...")

from google.colab import runtime
runtime.unassign()